For Tarla Dalal "Region" recipe

In [3]:
import json
import re

# Load JSON
with open("tarla_dalal_recipes_fixed.json", "r", encoding="utf-8") as f:
    data = json.load(f)

cleaned_data = []

for recipe in data:
    name = recipe.get("name", "").strip()
    ingredients = recipe.get("ingredients", [])
    instructions = recipe.get("instructions", [])
    region = recipe.get("region", "").strip()

    # Basic validation
    if not (name and ingredients and instructions):
        continue

    # Clean ingredients
    clean_ingredients = [re.sub(r"[^a-zA-Z0-9\s,]", "", ing.lower()).strip() for ing in ingredients]

    # Clean instructions
    clean_instructions = [re.sub(r"^\d+[\).]*\s*", "", step).strip() for step in instructions]

    cleaned_data.append({
        "name": name,
        "region": region,
        "ingredients": clean_ingredients,
        "instructions": clean_instructions
    })

# Deduplicate by name + region
unique = {(d["name"].lower(), d["region"].lower()): d for d in cleaned_data}
final_data = list(unique.values())

# Save
with open("cleaned_tarla_dalal.json", "w", encoding="utf-8") as f:
    json.dump(final_data, f, indent=2, ensure_ascii=False)

print(f"✅ Cleaned {len(final_data)} recipes saved to 'cleaned_tarla_dalal.json'")


✅ Cleaned 691 recipes saved to 'cleaned_tarla_dalal.json'


For Hebbbers Kitchen

In [5]:
import pandas as pd
import re

# Load CSV
df = pd.read_csv("indian_recipes_cleaned.csv")

# Drop rows with missing info
df.dropna(subset=["Dish Name", "Ingredients", "Instructions"], inplace=True)

# Clean function for ingredients
def clean_ingredients(ing):
    ing = ing.lower()
    ing = re.sub(r"[^a-zA-Z0-9\s,]", "", ing)
    return ing.strip()

# Clean function for instructions
def clean_instructions(instr):
    steps = instr.splitlines()
    clean_steps = [re.sub(r"^(step\s*\d+[:.)\-]*|\d+[:.)\-]*)\s*", "", step, flags=re.IGNORECASE).strip()
                   for step in steps if step.strip()]
    return clean_steps

# Apply cleaning
df["Cleaned Ingredients"] = df["Ingredients"].apply(clean_ingredients)
df["Cleaned Instructions"] = df["Instructions"].apply(clean_instructions)

# Standardize dish names
df["Dish Name"] = df["Dish Name"].str.strip().str.title()

# Deduplicate
df = df.drop_duplicates(subset=["Dish Name", "Cleaned Ingredients"])

# Reformat for saving
cleaned_df = df[["Dish Name", "Cleaned Ingredients", "Cleaned Instructions"]]
cleaned_df.columns = ["name", "ingredients", "instructions"]

# Save
cleaned_df.to_json("cleaned_hebbars_kitchen.json", orient="records", indent=2, force_ascii=False)
print("✅ Cleaned Hebbars Kitchen data saved to 'cleaned_hebbars_kitchen.json'")


✅ Cleaned Hebbars Kitchen data saved to 'cleaned_hebbars_kitchen.json'
